In [1]:
import requests
import re
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from selenium import webdriver
import time

In [2]:
url = 'https://app.thestorygraph.com/browse'
headers = {"User-Agent": "Mozilla/5.0"}  
response = requests.get(url, headers=headers)

print(response.status_code)

403


In [3]:
soup = BeautifulSoup(response.text, 'html.parser')
type(soup)

bs4.BeautifulSoup

In [13]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get("https://app.thestorygraph.com/browse")

time.sleep(3)

# Scroll multiple times
for i in range(20):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)

html = driver.page_source

from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")

books = soup.find_all("div", class_="book-pane-content")


for book in books:
    print(book.text.strip())

Project Hail Mary

Andy Weir



476 pages • hardcover • 2021
            

 • 159 editions


ISBN/UID:  9780593135204
Format: Hardcover
Language: English
Original Pub Year: 2021
Edition Pub Date: 04 May 2021
Publisher: Ballantine Books





fiction
science fiction
adventurous
funny
hopeful
medium-paced


fiction
science fiction
adventurous
funny
hopeful
medium-paced











                to read
 



Expand dropdown menu




read
currently reading
did not finish







           Remove Book
         

         You have tagged this book. Do you also want to remove the tags?
         

         Your reading history and any review will be deleted either way.
         


                   No, keep my tags
           

                   Yes, remove my tags
           


                 Cancel
         




           Remove Book
         

         Are you sure you want to remove this book from your lists?
         

         Your reading history and any review will be deleted.
   

In [ ]:
import time
import random
from bs4 import BeautifulSoup
import requests

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from tqdm import tqdm


data = []
seen = set()

books = soup.find_all("div", class_="book-pane-content")


for book in books:

    # TITLE
    title_tag = book.find("h3")
    title = title_tag.get_text(strip=True) if title_tag else None

    # AUTHOR
    author_tag = book.find("p", class_="font-body")
    author = author_tag.get_text(strip=True) if author_tag else None

    # SERIES (BOOLEAN)
    series_tag = book.find("p", class_="font-semibold")
    has_series = series_tag is not None

    # LINK (for dedup)
    link_tag = book.find("h3").find("a") if book.find("h3") else None
    link = link_tag["href"] if link_tag else None

    # PAGE COUNT + YEAR
    info_tag = book.find("p", class_="text-xs")

    pages = None
    year = None

    if info_tag:
        info_text = info_tag.get_text(" ", strip=True)

        parts = info_text.split("•")

        for part in parts:
            part = part.strip()

            if "pages" in part:
                pages = part.replace("pages", "").strip()

            elif part.isdigit():
                year = part

    # GENRES/TAGS
    genre_tags = book.select("div.book-pane-tag-section span")

    genres = [g.get_text(strip=True) for g in genre_tags]

    # Remove duplicates (because mobile + desktop versions both exist)
    genres = list(set(genres))

    # DEDUP
    if link not in seen:
        seen.add(link)

        data.append({
            "Title": title,
            "Author": author,
            "In A Series": has_series,
            "Pages": pages,
            "Year": year,
            "Genres/Tags": genres,
            "Link": link
        })


df = pd.DataFrame(data)

ratings = []
base_url = "https://app.thestorygraph.com"

for link in tqdm(df["Link"], desc="Scraping ratings"):
    full_url = base_url + link
    driver.get(full_url)

    try:
        container = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "div[aria-label*='Book rating']")
            )
        )
    
        aria = container.get_attribute("aria-label")
    
        import re
        match = re.search(r"Book rating:\s*([\d.]+)", aria)
        rating = match.group(1) if match else None

    except:
        rating = None
    ratings.append(rating)

df["Rating"] = ratings

print(df.head())
len(df)
driver.quit()

Scraping ratings: 100%|██████████| 210/210 [04:19<00:00,  1.24s/it]


                                         Title          Author  In A Series  \
0                   Project Hail MaryAndy Weir       Andy Weir        False   
1                In Her Own LeagueLiz Tomforde    Liz Tomforde         True   
2  This Story Might Save Your LifeTiffany Crum    Tiffany Crum        False   
3              The CorrespondentVirginia Evans  Virginia Evans        False   
4            Dungeon Crawler CarlMatt Dinniman   Matt Dinniman         True   

  Pages  Year                                        Genres/Tags  \
0   476  2021  [funny, adventurous, hopeful, science fiction,...   
1   440  2026  [sports, emotional, romance, lighthearted, fun...   
2   368  2026  [emotional, thriller, crime, tense, fiction, f...   
3   304  2025  [reflective, emotional, literary, hopeful, fic...   
4   446  2020  [fantasy, funny, adventurous, science fiction,...   

                                          Link Rating  
0  /books/ac3ea915-993d-4f30-8632-0f91e4ad0704    4.5  
1  /

In [15]:
#remove links from the dataframe
df = df.drop(columns=["Link"])

#Create a csv file of the data
df.to_csv("popularBooksList.csv", index=False)
